In [1]:
import pandas as pd
import numpy as np

accounts = pd.read_csv("../data/accounts.csv")
subscriptions = pd.read_csv("../data/subscriptions.csv")
feature_usage = pd.read_csv("../data/feature_usage.csv")
support_tickets = pd.read_csv("../data/support_tickets.csv")
churn_events = pd.read_csv("../data/churn_events.csv")

In [2]:
accounts.head()
subscriptions.head()
feature_usage.head()
support_tickets.head()
churn_events.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [3]:
accounts = accounts.drop_duplicates()
subscriptions = subscriptions.drop_duplicates()

In [4]:
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])

subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"])

In [5]:
subscriptions[~subscriptions.account_id.isin(accounts.account_id)]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [9]:
usage = feature_usage.merge(
    subscriptions[['subscription_id','account_id']],
    on='subscription_id',
    how='left'
)

In [10]:
usage.columns

Index(['usage_id', 'subscription_id', 'usage_date', 'feature_name',
       'usage_count', 'usage_duration_secs', 'error_count', 'is_beta_feature',
       'account_id'],
      dtype='object')

In [11]:
usage_summary = usage.groupby("account_id").agg(
    usage_count_total=("usage_count","sum"),
    feature_count=("feature_name","nunique")
).reset_index()

In [12]:
subs_agg = subscriptions.groupby("account_id").agg(
    total_mrr=("mrr_amount","sum"),
    total_arr=("arr_amount","sum"),
    active_subscriptions=("subscription_id","count"),
    total_seats=("seats","sum")
).reset_index()

In [13]:
support_agg = support_tickets.groupby("account_id").agg(
    ticket_count=("ticket_id","count"),
    avg_resolution_time=("resolution_time_hours","mean"),
    escalated_tickets=("escalation_flag","sum")
).reset_index()

In [14]:
support_agg.head()

,account_id,ticket_count,avg_resolution_time,escalated_tickets
0,A-00bed1,4,31.750000,0
1,A-00cac8,2,33.000000,0
2,A-0158bb,1,32.000000,0
3,A-016043,3,30.333333,0
4,A-019782,2,10.000000,0


In [15]:
churn_info = churn_events.groupby("account_id").agg(
    churn_date=("churn_date","max"),
    churn_reason=("reason_code", lambda x: x.mode()[0] if len(x) > 0 else None)
).reset_index()

In [16]:
analytics_base = accounts.copy()
analytics_base = analytics_base.merge(
    subs_agg,
    on="account_id",
    how="left"
)
analytics_base = analytics_base.merge(
    usage_summary,
    on="account_id",
    how="left"
)
analytics_base = analytics_base.merge(
    support_agg,
    on="account_id",
    how="left"
)
analytics_base = analytics_base.merge(
    churn_info,
    on="account_id",
    how="left"
)


In [17]:
analytics_base.fillna({
    "total_mrr":0,
    "total_arr":0,
    "usage_count_total":0,
    "feature_count":0,
    "ticket_count":0,
    "escalated_tickets":0
}, inplace=True)

In [18]:
analytics_base.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,...,total_arr,active_subscriptions,total_seats,usage_count_total,feature_count,ticket_count,avg_resolution_time,escalated_tickets,churn_date,churn_reason
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,151236,10,312,535,27,2.0,23.000000,0.0,2024-12-05,budget
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,...,120048,8,176,355,23,3.0,38.000000,0.0,NaN,NaN
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,219432,15,282,821,34,3.0,43.666667,0.0,2024-12-31,features
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False,...,111300,7,209,382,26,2.0,29.000000,0.0,2024-11-08,support
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True,...,585132,9,424,579,32,7.0,42.285714,1.0,2024-12-28,budget


In [19]:
analytics_base.shape

(500, 21)

In [20]:
analytics_base.to_csv("../data/analytics_base.csv", index=False)